# Predict diabetes subtype classification
HOMA1 estimates drived from insulin and c-peptide values are not directly interchangeable and may cause differences.

**Objective:** Evaluate performance of regression on downstream subtype assignment.

# Environment setup
Import required modules.

In [1]:
import numpy as np
import pandas as pd

import joblib

from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score
from sklearn.metrics import classification_report

Load pre-trained random forest model to assign clusters.

In [2]:
rf_path = "/data/user/brianlu/projects/precision_diabetes/cluster_prediction/exports/models/full_linear_rf_model.joblib"

rf = joblib.load(rf_path)

Load test data from HOMA1 regression.

In [3]:
data_path = "../exports/HOMA_regression/cluster_assignment_test_data.csv"

data = pd.read_csv(data_path)

Build label encoder to convert numeric cluster labels into named clusters.

In [4]:
# multiple classes to predict, transform into labels
le = LabelEncoder()
le.fit(np.array(["SAID", "SIDD", "SIRD", "MOD", "MARD"]))
le.classes_ = np.array(["SAID", "SIDD", "SIRD", "MOD", "MARD"])

# Data cleaning
Separate data into c-peptide reference set, insulin baselinse set, and fitted comparison set.

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166 entries, 0 to 165
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   DE_ID                      166 non-null    int64  
 1   source                     166 non-null    object 
 2   gender                     166 non-null    object 
 3   race                       166 non-null    object 
 4   gad                        166 non-null    int64  
 5   a1c                        166 non-null    float64
 6   bmi                        166 non-null    int64  
 7   diabetes_age               166 non-null    float64
 8   homa1_cpeptide_b           166 non-null    float64
 9   homa1_cpeptide_ir          166 non-null    float64
 10  homa1_insulin_b            166 non-null    float64
 11  homa1_insulin_ir           166 non-null    float64
 12  homa1_pred_b               166 non-null    float64
 13  homa1_pred_ir              166 non-null    float64

In [6]:
# HOMA1 from c-peptide as reference
data_cpeptide = data[["gad", "a1c", "bmi", "diabetes_age", "homa1_cpeptide_b", "homa1_cpeptide_ir"]]

# HOMA1 from insulin as baseline
data_insulin = data[["gad", "a1c", "bmi", "diabetes_age", "homa1_insulin_b", "homa1_insulin_ir"]]

## rename columns to match prediction model
data_insulin = data_insulin.rename(columns = {"homa1_insulin_b" : "homa1_cpeptide_b",
                                              "homa1_insulin_ir" : "homa1_cpeptide_ir"})
# HOMA1 from fitted insulin as comparison
data_fitted = data[["gad", "a1c", "bmi", "diabetes_age", "homa1_pred_b", "homa1_pred_ir"]]

## rename columns to match prediction model
data_fitted = data_fitted.rename(columns = {"homa1_pred_b" : "homa1_cpeptide_b",
                                              "homa1_pred_ir" : "homa1_cpeptide_ir"})

Separate data with cluster assignments known from [Lu et al](https://doi.org/10.1210/clinem/dgae516).

In [7]:
# UAB records with known clusters from hierarchical clustering
clustered_data = data.dropna(subset = "Ahlqvist").copy()

clustered_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166 entries, 0 to 165
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   DE_ID                      166 non-null    int64  
 1   source                     166 non-null    object 
 2   gender                     166 non-null    object 
 3   race                       166 non-null    object 
 4   gad                        166 non-null    int64  
 5   a1c                        166 non-null    float64
 6   bmi                        166 non-null    int64  
 7   diabetes_age               166 non-null    float64
 8   homa1_cpeptide_b           166 non-null    float64
 9   homa1_cpeptide_ir          166 non-null    float64
 10  homa1_insulin_b            166 non-null    float64
 11  homa1_insulin_ir           166 non-null    float64
 12  homa1_pred_b               166 non-null    float64
 13  homa1_pred_ir              166 non-null    float64

In [8]:
# HOMA1 from insulin as baseline
clustered_data_insulin = clustered_data[["gad", "a1c", "bmi", "diabetes_age", 
                                         "cluster_homa1_insulin_b", "cluster_homa1_insulin_ir"]]

## rename columns to match prediction model
clustered_data_insulin = clustered_data_insulin.rename(columns = {"cluster_homa1_insulin_b" : "homa1_cpeptide_b",
                                              "cluster_homa1_insulin_ir" : "homa1_cpeptide_ir"})
# HOMA1 from fitted insulin as comparison
clustered_data_fitted = clustered_data[["gad", "a1c", "bmi", "diabetes_age", 
                                        "cluster_homa1b_pred", "cluster_homa1ir_pred"]]

## rename columns to match prediction model
clustered_data_fitted = clustered_data_fitted.rename(columns = {"cluster_homa1b_pred" : "homa1_cpeptide_b",
                                              "cluster_homa1ir_pred" : "homa1_cpeptide_ir"})

# Model evaluation
Use trained model to assign clusters for each dataset.

## Model prediction against prediction by c-peptide

In [9]:
predictors = ["gad", "a1c", "bmi", "diabetes_age", "homa1_cpeptide_b", "homa1_cpeptide_ir"]

# assign cluster based on trained model
mapped_clusters = rf.predict(data_cpeptide[predictors])

# convert encoded labels back to cluster names
mapped_clusters = le.inverse_transform(mapped_clusters)

# insert back into data
data["cpeptide_clusters"] = mapped_clusters

In [10]:
# assign cluster based on trained model
mapped_clusters = rf.predict(data_insulin[predictors])

# convert encoded labels back to cluster names
mapped_clusters = le.inverse_transform(mapped_clusters)

# insert back into data
data["insulin_clusters"] = mapped_clusters

In [11]:
# assign cluster based on trained model
mapped_clusters = rf.predict(data_fitted[predictors])

# convert encoded labels back to cluster names
mapped_clusters = le.inverse_transform(mapped_clusters)

# insert back into data
data["fitted_clusters"] = mapped_clusters

In [12]:
scores = pd.DataFrame({"metric" : ["accuracy", "precision", "recall", "F1", "Kappa"],
                       "insulin HOMA1" : [balanced_accuracy_score(data["cpeptide_clusters"], data["insulin_clusters"]),
                                         precision_score(data["cpeptide_clusters"], data["insulin_clusters"], 
                                                                  average = "weighted"),
                                         recall_score(data["cpeptide_clusters"], data["insulin_clusters"], 
                                                                  average = "weighted"),
                                         f1_score(data["cpeptide_clusters"], data["insulin_clusters"], 
                                                                  average = "weighted"),
                                         cohen_kappa_score(data["cpeptide_clusters"], data["insulin_clusters"])],
                       "regression HOMA1" : [balanced_accuracy_score(data["cpeptide_clusters"], data["fitted_clusters"]),
                                         precision_score(data["cpeptide_clusters"], data["fitted_clusters"], 
                                                                  average = "weighted"),
                                         recall_score(data["cpeptide_clusters"], data["fitted_clusters"], 
                                                                  average = "weighted"),
                                         f1_score(data["cpeptide_clusters"], data["fitted_clusters"], 
                                                                  average = "weighted"),
                                         cohen_kappa_score(data["cpeptide_clusters"], data["fitted_clusters"])]})

scores

,metric,insulin HOMA1,regression HOMA1
0,accuracy,0.704005,0.895368
1,precision,0.866015,0.910740
2,recall,0.626506,0.891566
3,F1,0.641601,0.895480
4,Kappa,0.539095,0.859705


Export scores.

In [13]:
scores.to_csv("../exports/HOMA_regression/scores_against_cpeptide.csv", index = False)

## Model prediction against known clusters

In [14]:
clustered_data = clustered_data.rename(columns = {"Ahlqvist" : "hierarchical"})

In [15]:
# assign cluster based on trained model
mapped_clusters = rf.predict(clustered_data_insulin[predictors])

# convert encoded labels back to cluster names
mapped_clusters = le.inverse_transform(mapped_clusters)

# insert back into data
clustered_data["insulin_clusters"] = mapped_clusters

In [16]:
# assign cluster based on trained model
mapped_clusters = rf.predict(clustered_data_fitted[predictors])

# convert encoded labels back to cluster names
mapped_clusters = le.inverse_transform(mapped_clusters)

# insert back into data
clustered_data["fitted_clusters"] = mapped_clusters

In [17]:
scores = pd.DataFrame({"metric" : ["accuracy", "precision", "recall", "F1", "Kappa"],
                       "insulin HOMA1" : [balanced_accuracy_score(clustered_data["hierarchical"], clustered_data["insulin_clusters"]),
                                         precision_score(clustered_data["hierarchical"], clustered_data["insulin_clusters"], 
                                                                  average = "weighted"),
                                         recall_score(clustered_data["hierarchical"], clustered_data["insulin_clusters"], 
                                                                  average = "weighted"),
                                         f1_score(clustered_data["hierarchical"], clustered_data["insulin_clusters"], 
                                                                  average = "weighted"),
                                         cohen_kappa_score(clustered_data["hierarchical"], clustered_data["insulin_clusters"])],
                       "regression HOMA1" : [balanced_accuracy_score(clustered_data["hierarchical"], clustered_data["fitted_clusters"]),
                                         precision_score(clustered_data["hierarchical"], clustered_data["fitted_clusters"], 
                                                                  average = "weighted"),
                                         recall_score(clustered_data["hierarchical"], clustered_data["fitted_clusters"], 
                                                                  average = "weighted"),
                                         f1_score(clustered_data["hierarchical"], clustered_data["fitted_clusters"], 
                                                                  average = "weighted"),
                                         cohen_kappa_score(clustered_data["hierarchical"], clustered_data["fitted_clusters"])]})

scores

,metric,insulin HOMA1,regression HOMA1
0,accuracy,0.701369,0.877595
1,precision,0.865975,0.897436
2,recall,0.620482,0.873494
3,F1,0.633345,0.879255
4,Kappa,0.532227,0.836399


In [18]:
scores.to_csv("../exports/HOMA_regression/scores_against_hierarchical.csv", index = False)

In [19]:
from watermark import watermark

print(watermark())

Last updated: 2025-06-28T17:16:23.436154-05:00

Python implementation: CPython
Python version       : 3.12.6
IPython version      : 8.27.0

Compiler    : GCC 13.3.0
OS          : Linux
Release     : 3.10.0-1160.24.1.el7.x86_64
Machine     : x86_64
Processor   : x86_64
CPU cores   : 48
Architecture: 64bit

